In [70]:
# !pip install dlib pillow requests scipy tqdm
# Cell 2: Import necessary libraries
import scipy.stats  # For percentile calculations (normal distribution)
import os
import bz2
import requests
import numpy as np
import dlib
import PIL.Image
import scipy.ndimage
import matplotlib.pyplot as plt
import cv2
import numpy as np
from PIL import Image
from IPython.display import display, clear_output
import sys
from blur_wavelet import blur_detect as hwt_blur_detect
from collections import defaultdict
from scipy import stats


landmarks_model_path = "data_root/cache/ffhq/shape_predictor_68_face_landmarks.dat"


def pil_to_cv2(pil_img):
    """
    Convert a PIL Image to an OpenCV image (NumPy array in BGR format).
    Returns a new array; does not modify the original.
    """
    img = np.array(pil_img)
    if img.ndim == 2:  # Grayscale
        return img.copy()
    elif img.shape[2] == 4:  # RGBA
        return cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
    else:  # RGB
        return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

def is_hwt_blurry(img=None, img_path=None, threshold=35, min_zero=0.0000):

    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")
    per, blurext = hwt_blur_detect(processed_img, threshold)
    is_blur = per <= min_zero
    
    print(f"Blur detection percentage: {per:.5f}, is blurry: {is_blur}")
    return is_blur

    
def is_blurry(img=None, img_path=None, threshold=100):
    """
    Check if the image is blurry using Laplacian variance.
    Preserves the original image object.
    """
    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")

    gray = cv2.cvtColor(processed_img, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    print(f"Laplacian variance: {variance:.2f}")
    return variance < threshold

def is_bad_fsb_lighting(img=None, img_path=None, person_stats=None):
    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")
    
    results = process_fsb_lighting_face_image(processed_img)
    
    if results is None:
        print("❌⚠️  No face detected.")
        return True # bad lighting if no face detected
    fsb = results['fsb']
    
    
    print(f"FSB (Face Specific Brightness): {fsb:.2f}")
    
    if person_stats is not None:
        lower_bound, upper_bound = get_fsb_bounds(person_stats['race'], person_stats['gender'])
        # print(f"FSB bounds for {person_stats['race']} {person_stats['gender']}: {lower_bound:.2f} - {upper_bound:.2f}")
        if lower_bound <= fsb <= upper_bound:
            # print(" FSB is inside the acceptable range for this demographic (unexpected).")
            return False
        else:
            # print("⚠️ FSB is outside the acceptable range for this demographic (expected).")
            return True

    # print(f"FSB (Face Specific Brightness): {fsb:.2f}")
    

def is_bad_lighting(img=None, img_path=None, dark_thresh=30, bright_thresh=220):
    """
    Check if the image has bad lighting using mean grayscale intensity.
    Preserves the original image object.
    """
    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")

    gray = cv2.cvtColor(processed_img, cv2.COLOR_BGR2GRAY)
    mean_intensity = np.mean(gray)
    print(f"Mean intensity: {mean_intensity:.2f}")
    return (mean_intensity < dark_thresh) or (mean_intensity > bright_thresh)



In [83]:
data_info = {
    'obama': {'race': 'black', 'gender': 'male', 'unseen': False},
    'rihanna': {'race': 'black', 'gender': 'female', 'unseen': False},
    'edsheeran': {'race': 'white', 'gender': 'male', 'unseen': False},
    'mrobbie': {'race': 'white', 'gender': 'female', 'unseen': False},
    # 'osama': {'race': 'black', 'gender': 'male', 'seen': False},
    # 'honer': {'race': 'white', 'gender': 'male', 'seen': False},
    
    'asante': {'race': 'black', 'gender': 'male', 'unseen': True},
    'reese': {'race': 'black', 'gender': 'female', 'unseen': True},
    'nivola': {'race': 'white', 'gender': 'male','unseen': True},
    'earle': {'race': 'white', 'gender': 'female', 'unseen': True},

    'leowoodal': {'race': 'white', 'gender': 'male', 'unseen': True},
    'starkey': {'race': 'white', 'gender': 'male', 'unseen': True},
    'apierre': {'race': 'black', 'gender': 'male', 'unseen': True},
    'skyhblack': {'race': 'black', 'gender': 'male', 'unseen': True},

    'sophiewilde':{'race': 'black', 'gender': 'female', 'unseen': True},
    'edebiri':{'race': 'black', 'gender': 'female', 'unseen': True},
    'mmadison': {'race': 'white', 'gender': 'female', 'unseen': True},
    'nicoparker': {'race': 'white', 'gender': 'female', 'unseen': True},
    
    'chemsworth': {'race': 'white', 'gender': 'male', 'unseen': False},  # Chris Hemsworth
    'aadam': {'race': 'white', 'gender': 'female', 'unseen': False},       # Anne Adam
    'ahathaway': {'race': 'white', 'gender': 'female', 'unseen': False}, # Anne Hathaway
    'ajolie': {'race': 'white', 'gender': 'female', 'unseen': False},    # Angelina Jolie
    'amber': {'race': 'white', 'gender': 'female', 'unseen': False},     # Likely Amber Heard
    'cevans': {'race': 'white', 'gender': 'male', 'unseen': False},      # Chris Evans
    'agarfield': {'race': 'white', 'gender': 'male', 'unseen': False},   # Andrew Garfield
    'adriver': {'race': 'white', 'gender': 'male', 'unseen': False},     # Adam Driver
    'mcarey': {'race': 'black', 'gender': 'female', 'unseen': False},    # Mariah Carey (black heritage)
    'octavia': {'race': 'black', 'gender': 'female', 'unseen': False},   # Octavia Spencer
    'oprah': {'race': 'black', 'gender': 'female', 'unseen': False},     # Oprah Winfrey
    'morganf': {'race': 'black', 'gender': 'male', 'unseen': False},     # Morgan Freeman
    'drake': {'race': 'black', 'gender': 'male', 'unseen': False},       # Drake (mixed but usually listed as black)
    'idris': {'race': 'black', 'gender': 'male', 'unseen': False},       # Idris Elba
    
    
    
}

for concept in list(data_info.keys()):
    img_path = f"data_root/data/real_data/{concept}/aligned/{concept}-5-v0"
    if not os.path.exists(img_path):
        print(f"{img_path} does not exist!")
    else:
        print(len(os.listdir(img_path)), "images found in", img_path)
    data_info[concept]['img_dir'] = img_path

# Demographic FSB statistics (mean, std) from Wu et al. (2023)
demographic_fsb_stats = {
    ("black", "male"):   {"mean": 138.7, "std": 32.5},
    ("black", "female"): {"mean": 143.5, "std": 32.6},
    ("white", "male"):   {"mean": 188.3, "std": 25.6},
    ("white", "female"): {"mean": 191.4, "std": 25.0}
}

5 images found in data_root/data/real_data/obama/aligned/obama-5-v0
5 images found in data_root/data/real_data/rihanna/aligned/rihanna-5-v0
5 images found in data_root/data/real_data/edsheeran/aligned/edsheeran-5-v0
5 images found in data_root/data/real_data/mrobbie/aligned/mrobbie-5-v0
5 images found in data_root/data/real_data/asante/aligned/asante-5-v0
5 images found in data_root/data/real_data/reese/aligned/reese-5-v0
5 images found in data_root/data/real_data/nivola/aligned/nivola-5-v0
5 images found in data_root/data/real_data/earle/aligned/earle-5-v0
5 images found in data_root/data/real_data/leowoodal/aligned/leowoodal-5-v0
5 images found in data_root/data/real_data/starkey/aligned/starkey-5-v0
6 images found in data_root/data/real_data/apierre/aligned/apierre-5-v0
5 images found in data_root/data/real_data/skyhblack/aligned/skyhblack-5-v0
5 images found in data_root/data/real_data/sophiewilde/aligned/sophiewilde-5-v0
5 images found in data_root/data/real_data/edebiri/aligned/e

In [84]:
def copy_file(n=5,seed=0):
    for concept, info in data_info.items():
        src_aligned_dir = f'data_root/data/real_data/{concept}/raw/aligned/'
        dst_aligned_dir = f'data_root/data/real_data/{concept}/aligned/{concept}-5-v0'

        if not os.path.exists(dst_aligned_dir):
            os.makedirs(dst_aligned_dir)
        else:
            print(f"Directory {dst_aligned_dir} already exists, skipping creation.")

        # copy up to n files from src to dst (randomly select n files, seed for reproducibility)
        if not os.path.exists(src_aligned_dir):
            print(f"Source directory {src_aligned_dir} does not exist, skipping.")
            continue

        files = [f for f in os.listdir(src_aligned_dir) if os.path.isfile(os.path.join(src_aligned_dir, f))]
        if len(files) == 0:
            print(f"No files found in {src_aligned_dir}, skipping.")
            continue

        np.random.seed(seed)
        selected_files = np.random.choice(files, min(n, len(files)), replace=False)
        for file_name in selected_files:
            src_file = os.path.join(src_aligned_dir, file_name)
            dst_file = os.path.join(dst_aligned_dir, file_name)
            if not os.path.exists(dst_file):
                print(f"Copying {src_file} to {dst_file}")
                os.system(f'cp "{src_file}" "{dst_file}"')
            else:
                print(f"File {dst_file} already exists, skipping copy.")
# copy_file(n=5, seed=0)

In [78]:
# ==========================================
# 1. PATH CONFIGURATION - CRITICAL FIX
# ==========================================
BASE_DIR = "/home/nessessence/mnt_tl_vision16/home/nessessence/uul"
FACE_BRIGHTNESS_DIR = os.path.join(BASE_DIR, "FaceBrightness")
FACE_PARSING_DIR = os.path.join(FACE_BRIGHTNESS_DIR, "face_parsing")
# Clear conflicting paths and add correct ones
sys.path = [p for p in sys.path if "uul" not in p]
sys.path.insert(0, FACE_PARSING_DIR)
sys.path.insert(0, FACE_BRIGHTNESS_DIR)

print("✅ Python path configured:")
for p in sys.path[:2]:
    print(f"→ {p}")
# ==========================================
# 2. IMPORTS WITH VERIFICATION
# ==========================================
try:
    # Verify we're importing from correct location
    import face_parsing.test as face_test
    print(f"✓ Importing from: {face_test.__file__}")
    if not face_test.__file__.startswith(FACE_PARSING_DIR):
        raise ImportError("Wrong test.py being imported!")
    from face_parsing.test import evaluate
    from demographic_face_analyze import without_beard_region
    print("✓ All imports successful!")
    
except Exception as e:
    print(f"\n❌ Import Error: {e}")
    print("\n🔍 Debugging Info:")
    # Check test.py contents
    test_py_path = os.path.join(FACE_PARSING_DIR, "test.py")
    if os.path.exists(test_py_path):
        print(f"\nContents of {test_py_path}:")
        with open(test_py_path, 'r') as f:
            print(f.read(500))  # Show first 500 chars
        
    print("\n💡 Solutions:")
    print("1. Temporary rename conflicting file:")
    print(f"   !mv {BASE_DIR}/test.py {BASE_DIR}/test.py.BAK")
    print("2. Or use absolute import:")
    print("   from FaceBrightness.face_parsing.test import evaluate")
    raise

# ==========================================
# 3. CORE ANALYSIS FUNCTIONS
# ==========================================
def weights_calc(data):
    """Calculate weights for brightness values"""
    num_list = list(data.values())
    total = sum(num_list)
    return np.array([num/total for num in num_list])
def calculate_fsb(image):
    """Calculate Face Skin Brightness (FSB) as per paper"""
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    return np.mean(gray)
def calculate_bim(image):
    """Calculate Brightness Information Metric (BIM)"""
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Calculate histogram
    hist = cv2.calcHist([image], [0], None, [256], [0,256])
    hist /= hist.sum()  # Normalize
    
    level = np.arange(256)
    avg_brightness = np.sum(level * hist.flatten())
    bim = np.sum(np.abs(level - avg_brightness) * hist.flatten())
    return bim


def without_beard_region(image, mask, position=False, flatten=True):
    """
    Extract the upper face region (above the bottom of the nose).
    
    Args:
        image: Input BGR or grayscale image.
        mask: Face parsing mask.
        position: If True, also return pixel coordinates.
        flatten: If True, return 1D pixel values. If False, return a 2D (or 3D) masked image.
    
    Returns:
        Either a 1D array of pixel intensities or a masked image,
        and optionally pixel coordinates.
    """
    assert ((len(image.shape) == 3) or (len(image.shape) == 2)), f"Expect a gray or colored image, but got shape: {image.shape}"
    
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
    else:
        gray = image
        h, w = image.shape

    # Resize mask to match image size if needed
    if mask.shape[:2] != (h, w):
        mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
    else:
        mask = mask.astype(np.uint8)

    # Find all skin pixels (label 1)
    face_pos = np.where(mask == 1)

    # Find nose region (label 10)
    nose = np.where(mask == 10)
    if len(nose[0]) == 0:
        print("Nose not found in mask.")
        return None

    b_lim = nose[0][-1]  # bottom Y of the nose

    # Keep only skin pixels above bottom of nose
    mask_upper = np.zeros_like(mask, dtype=np.uint8)
    for y, x in zip(face_pos[0], face_pos[1]):
        if y < b_lim:
            mask_upper[y, x] = 1

    # Apply mask to grayscale image
    if flatten:
        pixel_values = gray[mask_upper == 1]
        return (pixel_values, np.where(mask_upper == 1)) if position else pixel_values
    else:
        masked_image = cv2.bitwise_and(gray, gray, mask=mask_upper)
        return (masked_image, np.where(mask_upper == 1)) if position else masked_image
    
def process_fsb_lighting_face_image(img):
    """Full processing pipeline for single image"""
    # Load image
    # img = cv2.imread(img_path)
    # if img is None:
    #     raise ValueError(f"Could not read image at {img_path}")
    # Get face mask
    mask = evaluate([img])[0]  # evaluate expects a list
    # print(f"Mask shape: {mask.shape}, unique values: {np.unique(mask)}")
    # Extract upper face region (excluding eyes, nose, lips etc.)
    upper_face = without_beard_region(img, mask)
    if upper_face is None:
        return None
    # print(f"Upper face : {upper_face}")
    
    # Calculate both metrics
    fsb = calculate_fsb(upper_face)
    bim = calculate_bim(upper_face)
    
    # for visualization, we can get the full masked image
    upper_face = without_beard_region(img, mask, flatten=False)  # Get full masked image
    
    
    return {
        'original': img,
        'mask': mask,
        'upper_face': upper_face,
        'fsb': fsb,
        'bim': bim
    }
# ==========================================
# 4. VISUALIZATION FUNCTIONS
# ==========================================
def display_results(results):
    """Display processing results with matplotlib"""
    # clear_output(wait=True)
    print(f"-FSB (Face Skin Brightness): {results['fsb']:.1f}/255")
    
    # Create figure
    plt.figure(figsize=(18, 5))
    
    # Original Image
    plt.subplot(1, 4, 1)
    plt.imshow(cv2.cvtColor(results['original'], cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis('off')
    
    # Face Mask
    plt.subplot(1, 4, 2)
    plt.imshow(results['mask'], cmap='tab20')
    plt.title("Face Parsing Mask")
    plt.axis('off')
    
    # Upper Face Region
    plt.subplot(1, 4, 3)
    plt.imshow(cv2.cvtColor(results['upper_face'], cv2.COLOR_BGR2RGB))
    plt.title("Upper Face Region")
    plt.axis('off')
    
    # Metrics display
    plt.subplot(1, 4, 4)
    plt.text(0.1, 0.7, f"FSB: {results['fsb']:.2f}\nBIM: {results['bim']:.2f}", 
             fontsize=12, bbox=dict(facecolor='white', alpha=0.5))
    plt.axis('off')
    plt.title("Brightness Metrics")
    
    plt.tight_layout()
    plt.show()
    
    # Brightness Histogram
    # gray = cv2.cvtColor(results['upper_face'], cv2.COLOR_BGR2GRAY) if len(results['upper_face'].shape) == 3 else results['upper_face']
    # plt.figure(figsize=(10, 4))
    # plt.hist(gray.ravel(), 256, [0,256])
    # plt.title("Pixel Brightness Distribution")
    # plt.xlabel("Brightness Value (0-255)")
    # plt.ylabel("Frequency")
    
    
    # print("\n📊 Interpretation:")
    # print(f"- FSB (Face Skin Brightness): {results['fsb']:.1f}/255")
    # print(f"- BIM (Brightness Information Metric): {results['bim']:.2f}")
    
    
    # Add FSB and optimal range markers
    # plt.axvline(x=results['fsb'], color='r', linestyle='--', label=f'FSB: {results["fsb"]:.1f}')
    # plt.axvspan(160, 205, alpha=0.2, color='green', label='Optimal Range (160-205)')
    # plt.legend()
    # plt.show()
    
    # Print interpretation

    # if results['fsb'] < 160:
    #     print("  ⚠️ Image may be UNDER-exposed (FSB < 160)")
    # elif results['fsb'] > 205:
    #     print("  ⚠️ Image may be OVER-exposed (FSB > 205)")
    # else:
    #     print("  ✅ Image is in optimal brightness range (160-205)")

# ==========================================
# 5. NOTEBOOK INTERFACE
# ==========================================
def analyze_fsb_lighting(img=None,img_path=None):
    """Complete analysis workflow"""
    if img is None and img_path is not None:
        processed_img = cv2.imread(img_path)
    elif isinstance(img, Image.Image):
        processed_img = pil_to_cv2(img)
    elif isinstance(img, np.ndarray):
        processed_img = img.copy()
    else:
        raise ValueError("You must provide either img or img_path.")
    try:
        results = process_fsb_lighting_face_image(processed_img)
        display_results(results)
        return results
    except Exception as e:
        print(f"❌ Analysis failed: {e}")
        # raise


def get_fsb_bounds(race, gender, lower_percentile=5, upper_percentile=95):
    stats = demographic_fsb_stats.get((race.lower(), gender.lower()))
    if not stats:
        raise ValueError(f"Demographic {(race, gender)} not found.")
    
    mean, std = stats["mean"], stats["std"]
    
    # Get z-scores for percentiles (using norm.ppf for standard normal)
    z_lower = scipy.stats.norm.ppf(lower_percentile / 100)
    z_upper = scipy.stats.norm.ppf(upper_percentile / 100)
    
    # print(z_lower, z_upper)
    
    # Your original equation
    lower_bound = mean + z_lower * std
    upper_bound = mean + z_upper * std
    
    return (lower_bound, upper_bound)

get_fsb_bounds("black", "male")


✅ Python path configured:
→ /home/nessessence/mnt_tl_vision16/home/nessessence/uul/FaceBrightness
→ /home/nessessence/mnt_tl_vision16/home/nessessence/uul/FaceBrightness/face_parsing
✓ Importing from: /home/nessessence/mnt_tl_vision16/home/nessessence/uul/FaceBrightness/face_parsing/test.py
✓ All imports successful!


(85.24225712407713, 192.15774287592285)

In [79]:
# DLib-based facial landmarks detector
class LandmarksDetector:
    def __init__(self, predictor_model_path):
        self.detector = dlib.get_frontal_face_detector()
        self.shape_predictor = dlib.shape_predictor(predictor_model_path)

    def get_landmarks(self, image_path):
        # img = dlib.load_rgb_image(image_path)
        
        # Instead of dlib.load_rgb_image()
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB if needed

        dets = self.detector(img, 1)
        for detection in dets:
            yield [(item.x, item.y) for item in self.shape_predictor(img, detection).parts()]

# FFHQ-style face alignment function
def image_align(src_file, face_landmarks, output_size=256, return_qsize=False):
    lm = np.array(face_landmarks)
    lm_eye_left = lm[36:42]
    lm_eye_right = lm[42:48]
    lm_mouth_outer = lm[48:60]

    # face_width = np.linalg.norm(lm[0] - lm[16])
    # print(f"Face width: {face_width:.2f} pixels")

    eye_left = np.mean(lm_eye_left, axis=0)
    eye_right = np.mean(lm_eye_right, axis=0)
    eye_avg = (eye_left + eye_right) * 0.5
    eye_to_eye = eye_right - eye_left
    mouth_left = lm_mouth_outer[0]
    mouth_right = lm_mouth_outer[6]
    mouth_avg = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg

    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    x *= max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])
    qsize = np.hypot(*x) * 2
    
    print("qsize:", qsize)

    img = PIL.Image.open(src_file).convert('RGB')
    transform_size = 4096
    img = img.transform((transform_size, transform_size), PIL.Image.QUAD,
                        (quad + 0.5).flatten(), PIL.Image.BILINEAR)
    img = img.resize((output_size, output_size), PIL.Image.Resampling.LANCZOS)
    if return_qsize: return img, qsize
    return img

# Main wrapper function for aligning and displaying
def align_and_display_face(img_path, output_size=512, return_filter=False):
    
    person_stats = None
    for data_label in data_info.keys():
        if data_label in img_path:
            person_stats = data_info[data_label]
            print(f"Processing {data_label} with stats: {person_stats}")
            break
    
    # if is_blurry(img_path=img_path):
    # if is_hwt_blurry(img_path=img_path):
    #     print("⚠️ Image rejected due to blur.")
    #     # return None
    # if is_bad_lighting(img_path=img_path):
    #     print("⚠️ Image rejected due to bad lighting.")
    #     # return None
    landmarks_list = list(detector.get_landmarks(img_path))
    if len(landmarks_list) == 0:
        print("❌ No face detected.")
        return None, None
    face_landmarks = landmarks_list[0]

    aligned_img, qsize = image_align(img_path, face_landmarks, output_size=output_size, return_qsize=True)
    # is bad resolution
    
    
    accept = True
    if qsize < 256:
        print("⚠️ Image rejected due to insufficient quality (qsize < 256).")
        # return None
        accept = False
    if is_blurry(img=aligned_img):  
        print("⚠️ Image rejected due to blur.")
        # return None
        accept = False
        
    if is_bad_fsb_lighting(img=aligned_img, person_stats=person_stats):
        print("⚠️ Image rejected due to bad FSB lighting.")
        accept = False
        
    analyze_fsb_lighting(img=aligned_img)
        # return None
    # is_bad_lighting(img=aligned_img)
    # (img=aligned_img)
    
    # if is_hwt_blurry(img=aligned_img):
    #     print("⚠️ Image rejected due to blur.")

    
    # Display both images
    orig = PIL.Image.open(img_path).convert('RGB')
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original")
    plt.imshow(orig)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Aligned")
    plt.imshow(aligned_img)
    plt.axis("off")
    plt.show()

    if return_filter:
        return aligned_img, accept
    return aligned_img


def align_face_with_filter(img_dir, is_save=True, delete_existing=True):
    save_dir = os.path.join(img_dir, "aligned")
    if delete_existing and os.path.exists(save_dir):
        print(f"Deleting existing directory: {save_dir}")
        for file in os.listdir(save_dir):
            file_path = os.path.join(save_dir, file)
            if os.path.isfile(file_path):
                os.remove(file_path)
            elif os.path.isdir(file_path):
                os.rmdir(file_path)
        os.rmdir(save_dir)
    if is_save and not os.path.exists(save_dir):
        os.makedirs(save_dir)

    for img_name in os.listdir(img_dir):
        if (img_name.lower().endswith(".jpg") or 
            img_name.lower().endswith(".png") or 
            img_name.lower().endswith(".webp") or 
            img_name.lower().endswith(".jpeg")) and not img_name.startswith('._'):
            
            full_img_path = os.path.join(img_dir, img_name)
            print(full_img_path)
            
            aligned_img, is_accept = align_and_display_face(full_img_path, return_filter=True)
            
            if is_save and aligned_img is not None and is_accept:
                # Change the extension to .png
                base_name = os.path.splitext(img_name)[0]
                save_path = os.path.join(save_dir, f"{base_name}.png")
                aligned_img.save(save_path, format='PNG')
                
                print(f"Saved aligned image to {save_path}")

detector = LandmarksDetector(landmarks_model_path)


In [81]:
global_stats = defaultdict(list)

for concept, concept_info in data_info.items():
    img_dir = concept_info['img_dir']
    for img_name in os.listdir(img_dir):
        if (img_name.lower().endswith(".jpg") or 
            img_name.lower().endswith(".png") or 
            img_name.lower().endswith(".webp") or 
            img_name.lower().endswith(".jpeg")) and not img_name.startswith('._'):
            
            full_img_path = os.path.join(img_dir, img_name)
            img = cv2.imread(full_img_path)
            results = process_fsb_lighting_face_image(img)


            fsb = results['fsb']
            
            if concept_info['unseen']:
                global_stats['unseen'] += [fsb]
            else:
                global_stats['seen'] += [fsb]
            
# Calculate basic statistics
seen_mean = np.mean(global_stats['seen'])
unseen_mean = np.mean(global_stats['unseen'])
mean_diff = seen_mean - unseen_mean

n_seen = len(global_stats['seen'])
n_unseen = len(global_stats['unseen'])
std_seen = np.std(global_stats['seen'], ddof=1)
std_unseen = np.std(global_stats['unseen'], ddof=1)

print("=== Descriptive Statistics ===")
print(f"Seen concepts (n={n_seen}): Mean FSB = {seen_mean:.2f}, SD = {std_seen:.2f}")
print(f"Unseen concepts (n={n_unseen}): Mean FSB = {unseen_mean:.2f}, SD = {std_unseen:.2f}")
print(f"Raw difference between means: {mean_diff:.2f}")

# Perform Welch's t-test (unpaired, unequal variances)
t_stat, p_value = stats.ttest_ind(global_stats['seen'], global_stats['unseen'], 
                                 equal_var=False, alternative='less')

# Calculate 95% confidence interval for the difference
se_diff = np.sqrt((std_seen**2/n_seen) + (std_unseen**2/n_unseen))
df = (std_seen**2/n_seen + std_unseen**2/n_unseen)**2 / ((std_seen**2/n_seen)**2/(n_seen-1) + (std_unseen**2/n_unseen)**2/(n_unseen-1))
conf_bound = mean_diff + stats.t.ppf(0.95, df) * se_diff

print("\n=== Hypothesis Test ===")
print(f"H₀: Mean difference (seen - unseen) ≥ 40")
print(f"H₁: Mean difference (seen - unseen) < 40")
print(f"t-statistic: {t_stat:.2f}, p-value: {p_value:.4f}")
print(f"Upper 95% confidence bound for difference: {conf_bound:.2f}")

# Conclusion
if conf_bound < 40:
    print("\nConclusion: The difference is statistically significantly less than 40 (p < 0.05)")
else:
    print("\nConclusion: Cannot conclude the difference is less than 40 (p ≥ 0.05)")

100%|████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.44it/s]


=== Descriptive Statistics ===
Seen concepts (n=89): Mean FSB = 152.81, SD = 24.71
Unseen concepts (n=60): Mean FSB = 152.35, SD = 24.22
Raw difference between means: 0.46

=== Hypothesis Test ===
H₀: Mean difference (seen - unseen) ≥ 40
H₁: Mean difference (seen - unseen) < 40
t-statistic: 0.11, p-value: 0.5445
Upper 95% confidence bound for difference: 7.21

Conclusion: The difference is statistically significantly less than 40 (p < 0.05)


In [69]:
for concept, concept_info in data_info.items():
    img_dir = f"data_root/data/real_data/{concept}/raw"
    align_face_with_filter(img_dir, is_save=True, delete_existing=True)


In [ ]:
for concept, concept_info in data_info.items():
    img_dir = f"data_root/data/real_data/{concept}/raw"
    align_face_with_filter(img_dir, is_save=True, delete_existing=True)


In [ ]:
img_dir = "data_root/data/real_data/edebiri/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/skyhblack/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/apierre/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/leowoodal/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/starkey/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/chemsworth/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/aadam/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/ahathaway/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/ajolie/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/amber/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/adriver/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/agarfield/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/cevans/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/oprah/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/octavia/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/mcarey/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/morganf/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/idris/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/drake/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/edsheeran/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/earle/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/reese/raw"
align_face_with_filter(img_dir,is_save=False)


In [ ]:
img_dir = "data_root/data/real_data/obama/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/asante/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/chavis/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/nivola/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/osama/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/honer/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/rihanna/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/obama/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/mrobbie/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/earle/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/reese/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/osama/raw"
align_face_with_filter(img_dir)


In [ ]:
img_dir = "data_root/data/real_data/edsheeran/raw"
align_face_with_filter(img_dir)


In [ ]:
dir_path = "data_root/data/real_data/rihanna/raw"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") or img_path.endswith(".webp") or  img_path.endswith(".jpeg") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:
dir_path = "data_root/data/real_data/obama/raw"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:
dir_path = "data_root/data/real_data/reese/raw"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") or img_path.endswith(".webp") or  img_path.endswith(".jpeg") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:
dir_path = "data_root/data/real_data/reese/raw"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") or img_path.endswith(".webp") or  img_path.endswith(".jpeg") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:
dir_path = "data_root/data/real_data/osama/raw"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") or img_path.endswith(".webp") or  img_path.endswith(".jpeg") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/rihanna/raw"
for img_path in os.listdir(dir_path):
        if (img_path.endswith(".jpg") or img_path.endswith(".png") or img_path.endswith(".webp") or  img_path.endswith(".jpeg") ) and not '._' in img_path:
                full_img_path = os.path.join(dir_path, img_path)
                aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/earle/raw"
for img_path in os.listdir(dir_path):
        if (img_path.endswith(".jpg") or img_path.endswith(".png") or img_path.endswith(".webp") or  img_path.endswith(".jpeg") ) and not '._' in img_path:
                full_img_path = os.path.join(dir_path, img_path)
                aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/resolution/"
for img_path in os.listdir(dir_path):
    if  (img_path.endswith(".jpeg")  or img_path.endswith(".jpg") or img_path.endswith(".png") ) and not '._' in img_path:
        full_img_path = os.path.j oin(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/lighting/"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/lighting/"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/obama/raw"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/mrobbie/raw"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/honer/face/honer-5-v0"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)


In [ ]:

dir_path = "data_root/data/real_data/reese/face/reese-5-v0"
for img_path in os.listdir(dir_path):
    if (img_path.endswith(".jpg") or img_path.endswith(".png") ) and not '._' in img_path:
        full_img_path = os.path.join(dir_path, img_path)
        aligned_img = align_and_display_face(full_img_path)
